In [1]:
import sys
print(sys.executable)

/home/valentino/miniconda3/envs/quantum_env/bin/python


In [2]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Audio, clear_output
from qutip import Bloch
import ipywidgets as widgets
from ipywidgets import interact

In [3]:
def visualize_and_play(theta_deg, phi_deg):
    clear_output(wait=True)

    # 1) Convertir a radianes
    theta = np.deg2rad(theta_deg)
    phi = np.deg2rad(phi_deg)

    # 2) Coordenadas cartesianas en la esfera de Bloch
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)

    # 3) Mapeo acústico estricto
    frequency = 200.0 + (800.0 - 200.0) * (theta / np.pi)
    amplitude = phi / (2.0 * np.pi)

    # 4) Síntesis de sonido
    sample_rate = 44100
    duration = 1.0
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    audio_signal = amplitude * np.sin(2 * np.pi * frequency * t)

    # 5) Esfera de Bloch
    fig = plt.figure(figsize=(6, 6))
    bloch = Bloch(fig=fig)
    bloch.add_vectors([x, y, z])
    bloch.vector_color = ['red']
    bloch.vector_width = 3
    bloch.xlabel = ['|+⟩', '|-⟩']
    bloch.ylabel = ['|+i⟩', '|-i⟩']
    bloch.zlabel = ['|0⟩', '|1⟩']
    bloch.show()

    # 6) Info
    print(f"theta = {theta_deg:.1f}°   |   phi = {phi_deg:.1f}°")
    print(f"x = {x:.4f}, y = {y:.4f}, z = {z:.4f}")
    print(f"frequency = {frequency:.2f} Hz")
    print(f"amplitude = {amplitude:.4f}")

    # 7) Audio
    display(Audio(audio_signal, rate=sample_rate, autoplay=False))

    plt.close(fig)

In [4]:
interact(
    visualize_and_play,
    theta_deg=widgets.IntSlider(
        value=90,
        min=0,
        max=180,
        step=1,
        description='θ (deg)'
    ),
    phi_deg=widgets.IntSlider(
        value=180,
        min=0,
        max=360,
        step=1,
        description='φ (deg)'
    )
);

interactive(children=(IntSlider(value=90, description='θ (deg)', max=180), IntSlider(value=180, description='φ…

In [5]:
%pip install imageio-ffmpeg

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import subprocess
import tempfile
import numpy as np
import matplotlib as mpl
import imageio_ffmpeg

from matplotlib.animation import FuncAnimation, FFMpegWriter
from scipy.io.wavfile import write as write_wav
from IPython.display import Video, display

In [7]:
def bloch_xyz(theta, phi):
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)
    return x, y, z


def smoothstep(s):
    return 3*s**2 - 2*s**3


def build_trajectory(
    theta_start_deg=90,
    phi_start_deg=180,
    target='0',
    duration=4.0,
    fps=24,
    phi_end_deg=None,
    phi_turns=0.0
):
    theta0 = np.deg2rad(theta_start_deg)
    phi0 = np.deg2rad(phi_start_deg % 360)
    theta_target = 0.0 if str(target) == '0' else np.pi

    n_frames = max(2, int(duration * fps))
    u = np.linspace(0.0, 1.0, n_frames)
    eased = smoothstep(u)

    theta_track = theta0 + (theta_target - theta0) * eased

    if phi_end_deg is None:
        phi_track = phi0 + 2 * np.pi * phi_turns * eased
    else:
        phi1 = np.deg2rad(phi_end_deg % 360)
        dphi = (phi1 - phi0 + np.pi) % (2 * np.pi) - np.pi
        phi_track = phi0 + (dphi + 2 * np.pi * phi_turns) * eased

    phi_track = np.mod(phi_track, 2 * np.pi)
    return theta_track, phi_track


def synth_dynamic_audio(theta_track, phi_track, duration, sample_rate=44100, amplitude_mode="pole"):
    n_samples = int(sample_rate * duration)

    frame_u = np.linspace(0.0, 1.0, len(theta_track))
    audio_u = np.linspace(0.0, 1.0, n_samples, endpoint=False)

    theta_audio = np.interp(audio_u, frame_u, theta_track)
    phi_audio = np.interp(audio_u, frame_u, phi_track)

    # Pitch dinámico desde theta
    freq = 200.0 + 600.0 * (theta_audio / np.pi)

    # Volumen dinámico
    if amplitude_mode == "phi":
        amp = phi_audio / (2.0 * np.pi)
    elif amplitude_mode == "pole":
        amp = 0.15 + 0.85 * np.abs(np.cos(theta_audio))
    else:
        raise ValueError("amplitude_mode debe ser 'pole' o 'phi'")

    # Señal mono base
    phase = 2.0 * np.pi * np.cumsum(freq) / sample_rate
    mono = amp * np.sin(phase)

    # Paneo estéreo desde phi
    # cos(phi) = proyección izquierda-derecha del azimut
    # +1 -> derecha, -1 -> izquierda
    pan = np.cos(phi_audio)

    # Equal-power panning para que no se caiga el volumen al centro
    pan_angle = (pan + 1.0) * (np.pi / 4.0)
    left = mono * np.cos(pan_angle)
    right = mono * np.sin(pan_angle)

    stereo = np.column_stack([left, right])

    # Fade corto para evitar clics
    fade_n = min(int(0.02 * sample_rate), n_samples // 2)
    if fade_n > 0:
        fade_in = np.linspace(0.0, 1.0, fade_n)
        fade_out = np.linspace(1.0, 0.0, fade_n)
        stereo[:fade_n, 0] *= fade_in
        stereo[:fade_n, 1] *= fade_in
        stereo[-fade_n:, 0] *= fade_out
        stereo[-fade_n:, 1] *= fade_out

    max_abs = np.max(np.abs(stereo))
    if max_abs > 0:
        stereo = 0.95 * stereo / max_abs

    return stereo.astype(np.float32)


def draw_bloch_frame(ax, theta, phi, target='0'):
    ax.cla()

    # Esfera
    u = np.linspace(0, 2*np.pi, 60)
    v = np.linspace(0, np.pi, 30)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, rstride=4, cstride=4, alpha=0.18, linewidth=0.7)

    # Ejes
    ax.plot([-1, 1], [0, 0], [0, 0], linewidth=1)
    ax.plot([0, 0], [-1, 1], [0, 0], linewidth=1)
    ax.plot([0, 0], [0, 0], [-1, 1], linewidth=1)

    # Ecuador
    eq = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(eq), np.sin(eq), np.zeros_like(eq), linewidth=1, alpha=0.25)

    # Etiquetas
    ax.text(1.10, 0, 0, '|+⟩', fontsize=11)
    ax.text(-1.18, 0, 0, '|-⟩', fontsize=11)
    ax.text(0, 1.10, 0, '|+i⟩', fontsize=11)
    ax.text(0, -1.18, 0, '|-i⟩', fontsize=11)
    ax.text(0, 0, 1.16, '|0⟩', fontsize=12)
    ax.text(0, 0, -1.22, '|1⟩', fontsize=12)

    # Vector del estado
    x, y, z = bloch_xyz(theta, phi)
    ax.quiver(0, 0, 0, x, y, z, color='crimson', linewidth=3, arrow_length_ratio=0.12)
    ax.scatter([x], [y], [z], s=45, color='crimson')

    # Marca del objetivo
    if str(target) == '0':
        ax.scatter([0], [0], [1], s=90, color='limegreen')
    else:
        ax.scatter([0], [0], [-1], s=90, color='limegreen')

    ax.set_xlim([-1.15, 1.15])
    ax.set_ylim([-1.15, 1.15])
    ax.set_zlim([-1.15, 1.15])
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=20, azim=35)
    ax.set_axis_off()


def make_qubit_video_to_basis(
    theta_start_deg=90,
    phi_start_deg=180,
    target='0',
    duration=4.0,
    fps=24,
    sample_rate=44100,
    amplitude_mode="pole",
    phi_end_deg=None,
    phi_turns=0.0,
    output_path="qubit_dynamic.mp4"
):
    theta_track, phi_track = build_trajectory(
    theta_start_deg=theta_start_deg,
    phi_start_deg=phi_start_deg,
    target=target,
    duration=duration,
    fps=fps,
    phi_end_deg=phi_end_deg,
    phi_turns=phi_turns
)

    audio_signal = synth_dynamic_audio(
        theta_track=theta_track,
        phi_track=phi_track,
        duration=duration,
        sample_rate=sample_rate,
        amplitude_mode=amplitude_mode
    )

    ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
    mpl.rcParams["animation.ffmpeg_path"] = ffmpeg_path

    output_path = os.path.abspath(output_path)

    with tempfile.TemporaryDirectory() as tmpdir:
        silent_video = os.path.join(tmpdir, "silent.mp4")
        wav_audio = os.path.join(tmpdir, "audio.wav")

        write_wav(wav_audio, sample_rate, audio_signal)

        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='3d')

        def update(frame_idx):
            theta = theta_track[frame_idx]
            phi = phi_track[frame_idx]

            draw_bloch_frame(ax, theta, phi, target=target)

            freq = 200.0 + 600.0 * (theta / np.pi)
            amp_phi = phi / (2.0 * np.pi)
            amp_pole = 0.15 + 0.85 * abs(np.cos(theta))
            amp_show = amp_phi if amplitude_mode == "phi" else amp_pole

            ax.set_title(
                f"Target |{target}⟩   |   θ = {np.rad2deg(theta):.1f}°   |   "
                f"f = {freq:.1f} Hz   |   A = {amp_show:.2f}",
                pad=18
            )
            return []

        anim = FuncAnimation(
            fig,
            update,
            frames=len(theta_track),
            interval=1000 / fps,
            blit=False
        )

        writer = FFMpegWriter(fps=fps, bitrate=2000)
        anim.save(silent_video, writer=writer, dpi=140)
        plt.close(fig)

        cmd = [
            ffmpeg_path,
            "-y",
            "-i", silent_video,
            "-i", wav_audio,
            "-c:v", "copy",
            "-c:a", "aac",
            "-b:a", "192k",
            "-shortest",
            "-movflags", "faststart",
            output_path
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError("ffmpeg no pudo unir video y audio.")

    display(Video(output_path, embed=True))
    print(f"Video guardado en: {output_path}")
    return output_path

In [8]:
make_qubit_video_to_basis(
    theta_start_deg=120,
    phi_start_deg=20,
    phi_end_deg=320,
    phi_turns=0.5,
    target='0',
    duration=5.0,
    fps=24,
    amplitude_mode="pole",
    output_path="qubit_hacia_0_con_paneo.mp4"
)

Video guardado en: /home/valentino/sonificacion_1/qubit_hacia_0_con_paneo.mp4


'/home/valentino/sonificacion_1/qubit_hacia_0_con_paneo.mp4'

In [9]:
make_qubit_video_to_basis(
    theta_start_deg=60,
    phi_start_deg=300,
    target='1',
    duration=4.0,
    fps=24,
    amplitude_mode="pole",
    output_path="qubit_hacia_1_paneo_fijo.mp4"
)

Video guardado en: /home/valentino/sonificacion_1/qubit_hacia_1_paneo_fijo.mp4


'/home/valentino/sonificacion_1/qubit_hacia_1_paneo_fijo.mp4'